# Model Confidence Set and sub-period losses

Model Confidence Set (MCS) of Hansen, Lunde and Nason (2011), computed with the `arch` package (`pip install arch`). The loss series for each model is the **daily cross-sectional average loss** over the asset-dates on which every model in the set has a valid forecast, so all models are compared on an identical sample.

Settings: size 10% (the 90% MCS), the $T_{\max}$ statistic (`method="max"`), stationary bootstrap with 10,000 replications. The bootstrap block length is set from the data: the largest Politis-White optimal stationary-bootstrap block length across the models' loss differentials (relative to the cross-model mean), rounded up.

The model set is the seven LSTMs: the four fixed lookbacks, LSTM_VAL, LSTM_PLACEBO (one lookback path shared by all stocks) and LSTM_AIC. MSE is computed on variances. Models that have not been run yet are skipped with a note.

The notebook also reports mean QLIKE and MSE (averaged through time within each stock, then across stocks, as in the paper) and realised utility (averaged across stocks each day, then through time) within the pre-specified sub-periods.

In [ ]:
from pathlib import Path
import math
import re

import numpy as np
import pandas as pd
from IPython.display import display
from arch.bootstrap import MCS, optimal_block_length

MAIN_MODELS = ["LSTM_10", "LSTM_21", "LSTM_42", "LSTM_63", "LSTM_VAL", "LSTM_AIC"]

MCS_SIZE = 0.10
MCS_REPS = 10_000
MCS_METHOD = "max"
MCS_SEED = 20261028
MIN_PERIOD_DATES = 60             # skip a sub-period with fewer test dates

ESTIMATE_TYPE = "QMLE-Trade"
VALUE_COLUMN = "Volatility"
DUPLICATE_POLICY = "drop"         # must match duplicate_policy in the model notebooks
SHARPE_RATIO = 0.4
RISK_AVERSION = 2.0

# Pre-specified sub-periods (test dates). "Other years" is every test date
# outside the three named periods.
PERIODS = {
    "Full sample": None,
    "Early sample (2004-2006)": [("2004-01-01", "2006-12-31")],
    "Financial crisis (2008-2009)": [("2008-01-01", "2009-12-31")],
    "COVID-19 (2020)": [("2020-01-01", "2020-12-31")],
    "Other years": "complement",
}

pd.set_option("display.width", 160)

In [ ]:
def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / "Results").is_dir() and (candidate / "Data").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate a project root containing Results/ and Data/.")


def natural_key(name: str):
    return [int(part) if part.isdigit() else part.lower() for part in re.split(r"(\d+)", name)]


def discover_models(results_dir: Path, models=None, include_robustness=False):
    """Every Results/<name>/<name>.csv. Robustness runs (names ending in
    _refit<N>m, _aic<N>y, _resel<N>m or _seed<N>) are excluded unless
    requested or listed explicitly."""
    found = {}
    for folder in results_dir.iterdir():
        main_file = folder / f"{folder.name}.csv"
        if folder.is_dir() and main_file.is_file():
            found[folder.name] = main_file
    if models is not None:
        missing = [m for m in models if m not in found]
        if missing:
            print(f"Not yet run (skipped): {missing}")
        found = {m: found[m] for m in models if m in found}
    elif not include_robustness:
        found = {k: v for k, v in found.items()
                 if not re.search(r"_(refit\d+m|aic\d+y|resel\d+m|seed\d+)", k)}
    return dict(sorted(found.items(), key=lambda item: natural_key(item[0])))


def parse_source_dates(values):
    """Same date parser as NN/NN.ipynb."""
    text = values.astype(str).str.strip()
    slash_mask = text.str.contains("/", regex=False)
    parsed = pd.Series(pd.NaT, index=text.index, dtype="datetime64[ns]")
    parsed.loc[~slash_mask] = pd.to_datetime(text.loc[~slash_mask], format="%Y-%m-%d", errors="raise")
    parsed.loc[slash_mask] = pd.to_datetime(text.loc[slash_mask], dayfirst=True, errors="raise")
    return parsed


def load_prediction_file(path: Path) -> pd.DataFrame:
    frame = pd.read_csv(path)
    if "date" not in frame.columns:
        raise ValueError(f"Missing date column in {path}")
    frame["date"] = pd.to_datetime(frame["date"], errors="raise")
    if frame["date"].duplicated().any():
        raise ValueError(f"Duplicate dates in {path}")
    frame = frame.set_index("date").sort_index()
    return frame.apply(pd.to_numeric, errors="coerce")


def load_actuals(tickers, data_dir: Path) -> pd.DataFrame:
    """Realised volatility, constructed exactly as in the model notebooks:
    same estimate type, same date parser, same duplicate policy."""
    series = []
    for ticker in tickers:
        path = data_dir / f"{ticker}.csv"
        if not path.is_file():
            raise FileNotFoundError(f"No realised-volatility file for {ticker}: {path}")
        frame = pd.read_csv(path)
        frame = frame.loc[frame["Type"].eq(ESTIMATE_TYPE), ["Date", VALUE_COLUMN]].copy()
        if frame.empty:
            raise ValueError(f"No {ESTIMATE_TYPE!r} observations in {path}")
        frame["Date"] = parse_source_dates(frame["Date"])
        duplicated = frame["Date"].duplicated(keep=False)
        if duplicated.any():
            dates = sorted(frame.loc[duplicated, "Date"].dt.date.unique())
            if DUPLICATE_POLICY == "drop":
                frame = frame.loc[~duplicated]
                action = "treated as missing"
            elif DUPLICATE_POLICY == "first":
                frame = frame.drop_duplicates("Date", keep="first")
                action = "first observation kept"
            else:
                raise ValueError("DUPLICATE_POLICY must be 'drop' or 'first'")
            print(f"Data note - {ticker}: repeated observations on {len(dates)} date(s) {action}: {dates}")
        item = pd.to_numeric(frame[VALUE_COLUMN], errors="coerce")
        item.index = frame["Date"]
        item.name = ticker
        series.append(item)
    return pd.concat(series, axis=1, sort=True)

def period_mask(dates, period):
    """Boolean mask over a DatetimeIndex for one entry of PERIODS."""
    dates = pd.DatetimeIndex(dates)
    if period is None:
        return np.ones(len(dates), dtype=bool)
    if isinstance(period, str) and period == "complement":
        named = np.zeros(len(dates), dtype=bool)
        for other in PERIODS.values():
            if isinstance(other, list):
                named |= period_mask(dates, other)
        return ~named
    mask = np.zeros(len(dates), dtype=bool)
    for start, end in period:
        mask |= (dates >= pd.Timestamp(start)) & (dates <= pd.Timestamp(end))
    return mask


def loss_matrix(actual, predicted, loss):
    if loss == "mse":
        # Squared error of the variance forecast.
        return np.square(np.square(actual) - np.square(predicted))
    if loss == "qlike":
        ratio = np.square(actual) / np.square(predicted)
        return ratio - np.log(ratio) - 1.0
    raise ValueError(loss)


def realised_utility(actual, predicted):
    ratio = actual / predicted
    return (SHARPE_RATIO**2 / RISK_AVERSION) * ratio - (SHARPE_RATIO**2 / (2 * RISK_AVERSION)) * ratio**2


ROOT = find_project_root(Path.cwd())
RESULTS_DIR = ROOT / "Results"
DATA_DIR = ROOT / "Data"
OUTPUT_DIR = RESULTS_DIR / "Analysis" / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

available = discover_models(RESULTS_DIR, include_robustness=True)
print(f"Available model folders: {', '.join(available)}")

In [ ]:
def load_set(model_names):
    """Predictions and actuals on the asset-dates where every model is valid."""
    present = [m for m in model_names if m in available]
    missing = [m for m in model_names if m not in available]
    if missing:
        print(f"Skipping models not yet run: {missing}")
    if len(present) < 2:
        raise ValueError(f"Need at least two models; found {present}")
    preds = {m: load_prediction_file(available[m]) for m in present}
    tickers = sorted(set.intersection(*(set(p.columns) for p in preds.values())))
    dates = preds[present[0]].index
    for p in preds.values():
        dates = dates.intersection(p.index)
    actual = load_actuals(tickers, DATA_DIR)
    dates = dates.intersection(actual.index).sort_values()
    y = actual.loc[dates, tickers].to_numpy(dtype=float)
    arrays = {m: preds[m].loc[dates, tickers].to_numpy(dtype=float) for m in present}
    valid = np.isfinite(y) & (y > 0)
    for a in arrays.values():
        valid &= np.isfinite(a) & (a > 0)
    keep = valid.any(axis=1)
    return {
        "models": present, "dates": dates[keep], "tickers": tickers,
        "y": y[keep], "pred": {m: a[keep] for m, a in arrays.items()}, "valid": valid[keep],
    }


def daily_average_losses(data, loss):
    """T x k frame: cross-sectional average loss on each date (common sample)."""
    out = {}
    counts = data["valid"].sum(axis=1)
    for m in data["models"]:
        values = np.where(data["valid"], loss_matrix(data["y"], data["pred"][m], loss), 0.0)
        out[m] = values.sum(axis=1) / counts
    return pd.DataFrame(out, index=data["dates"])


def run_mcs(losses):
    """90% MCS with the T_max statistic and a data-driven block length."""
    differentials = losses.sub(losses.mean(axis=1), axis=0)
    block = optimal_block_length(differentials)["stationary"].max()
    block = max(1, int(math.ceil(block)))
    mcs = MCS(losses, size=MCS_SIZE, reps=MCS_REPS, block_size=block,
              method=MCS_METHOD, bootstrap="stationary", seed=MCS_SEED)
    mcs.compute()
    pvalues = mcs.pvalues["Pvalue"].reindex(losses.columns)
    return pvalues, block

In [ ]:
mcs_rows = []
for set_name, model_names in [("LSTMs", MAIN_MODELS)]:
    if sum(m in available for m in model_names) < 2:
        print(f"{set_name} set: fewer than two of {model_names} have been run; skipped.")
        continue
    data = load_set(model_names)
    for loss in ["qlike", "mse"]:
        daily = daily_average_losses(data, loss)
        for period_name, period in PERIODS.items():
            losses = daily.loc[period_mask(daily.index, period)]
            if len(losses) < MIN_PERIOD_DATES:
                print(f"{set_name}, {loss.upper()}, {period_name}: only {len(losses)} dates; skipped.")
                continue
            pvalues, block = run_mcs(losses)
            for model, p in pvalues.items():
                mcs_rows.append({
                    "set": set_name, "loss": loss.upper(), "period": period_name,
                    "model": model, "mcs_pvalue": float(p), "in_90pct_mcs": bool(p >= MCS_SIZE),
                    "mean_daily_loss": float(losses[model].mean()),
                    "dates": int(len(losses)), "block_length": block,
                })
mcs_table = pd.DataFrame(mcs_rows)
mcs_table.to_csv(OUTPUT_DIR / "mcs_pvalues.csv", index=False)

for (set_name, loss), group in mcs_table.groupby(["set", "loss"], sort=False):
    print(f"\n{set_name} set - {loss}: MCS p-values (* = in the 90% MCS)")
    cells = group.assign(cell=lambda g: g["mcs_pvalue"].map("{:.3f}".format) + np.where(g["in_90pct_mcs"], "*", ""))
    display(cells.pivot(index="model", columns="period", values="cell").reindex(columns=list(PERIODS)))

In [ ]:
# Sub-period losses on the common sample of the LSTMs.
data = load_set(MAIN_MODELS)
rows = []
for period_name, period in PERIODS.items():
    in_period = period_mask(data["dates"], period)
    for m in data["models"]:
        row = {"period": period_name, "model": m}
        for loss in ["qlike", "mse"]:
            values = loss_matrix(data["y"], data["pred"][m], loss)
            stock_means = []
            for j in range(len(data["tickers"])):
                mask = in_period & data["valid"][:, j]
                if mask.any():
                    stock_means.append(values[mask, j].mean())
            row[loss.upper()] = float(np.mean(stock_means)) if stock_means else np.nan
        utility = np.where(data["valid"], realised_utility(data["y"], data["pred"][m]), 0.0)
        daily_ru = utility.sum(axis=1) / data["valid"].sum(axis=1)
        row["RU (% wealth)"] = float(daily_ru[in_period].mean() * 100) if in_period.any() else np.nan
        rows.append(row)
period_losses = pd.DataFrame(rows)
period_losses.to_csv(OUTPUT_DIR / "losses_by_period.csv", index=False)
for metric in ["QLIKE", "MSE", "RU (% wealth)"]:
    print(f"\n{metric}")
    display(period_losses.pivot(index="model", columns="period", values=metric).reindex(columns=list(PERIODS)))